# Dataset-2 (20 Newsgroups) — BERT Variation Experiment

**BERT Variation:** DistilBERT (`distilbert-base-uncased`)

This notebook uses the uploaded Dataset-2 CSV and follows the instructor's 8 combinations exactly.

Fixed settings: Epochs=5 with early stopping, Warmup Ratio=0.1, Max Sequence Length=128, Dropout=0.1, Optimizer=AdamW.

The best combination is selected using validation F1, and the selected model is evaluated once on the untouched test set.


In [ ]:
# FIX — Restore Colab-compatible scientific packages

!pip install -q --force-reinstall \
    "numpy==2.2.6" \
    "scipy==1.15.3" \
    "scikit-learn==1.6.1" \
    "pandas==2.2.3"

In [ ]:
!pip install -q -U transformers datasets accelerate

In [ ]:
# TEST — Check package compatibility

import numpy as np
import scipy
import sklearn
import pandas as pd

print("NumPy      :", np.__version__)
print("SciPy      :", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas     :", pd.__version__)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("\n✅ sklearn import successful!")

NumPy      : 2.2.6
SciPy      : 1.15.3
Scikit-learn: 1.6.1
Pandas     : 2.2.3

✅ sklearn import successful!


In [ ]:
# CELL 2 — Imports
import os, gc, json, random, math, inspect
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

print("Transformers version:", __import__("transformers").__version__)
print("PyTorch version:", torch.__version__)

Transformers version: 5.16.1
PyTorch version: 2.11.0+cu128


In [ ]:
# CELL 3 — GPU check
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Enable GPU in Colab.")


CUDA available: True
GPU: Tesla T4


In [ ]:
# CELL 4 — Reproducibility
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Seed:", SEED)


Seed: 42


In [ ]:
# CELL 5 — Load the uploaded Dataset-2
DATASET_PATH = '/content/drive/MyDrive/NLP dataset 2/20newsgroups_dataset.csv'

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"File not found: {DATASET_PATH}")

df = pd.read_csv(DATASET_PATH)

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


Original shape: (18846, 2)
Columns: ['text', 'category']


,text,category
0,\n\nI am sure some bashers of Pens fans are pr...,rec.sport.hockey
1,My brother is in the market for a high-perform...,comp.sys.ibm.pc.hardware
2,\n\n\n\n\tFinally you said what you dream abou...,talk.politics.mideast
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,comp.sys.ibm.pc.hardware
4,1) I have an old Jasmine drive which I cann...,comp.sys.mac.hardware


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# CELL 6 — Validate required columns
required_columns = {"text", "category"}
missing = required_columns - set(df.columns)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Required columns found:", required_columns)


Required columns found: {'text', 'category'}


In [ ]:
# CELL 7 — Clean data and select the four required categories
df = df.dropna(subset=["text", "category"]).copy()
df = df.drop_duplicates().copy()

print("Shape after cleaning:", df.shape)

selected_categories = [
    "comp.graphics",
    "rec.sport.hockey",
    "sci.space",
    "talk.politics.misc"
]

df = df[df["category"].isin(selected_categories)].copy()
df = df.reset_index(drop=True)

print("\nSelected Categories:")
print(df["category"].value_counts())

print("\nTotal Samples:", len(df))


Shape after cleaning: (18351, 2)

Selected Categories:
category
rec.sport.hockey      979
sci.space             959
comp.graphics         955
talk.politics.misc    760
Name: count, dtype: int64

Total Samples: 3653


In [ ]:
# CELL 8 — Define X and y
X = df["text"].astype(str)
y = df["category"].astype(str)

print("Number of texts:", len(X))
print("Number of labels:", len(y))


Number of texts: 3653
Number of labels: 3653


In [ ]:
# CELL 9 — Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Label Mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(i, "=", label)

NUM_LABELS = len(label_encoder.classes_)


Label Mapping:
0 = comp.graphics
1 = rec.sport.hockey
2 = sci.space
3 = talk.politics.misc


In [ ]:
# CELL 10 — 80/20 stratified train-test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training portion:", len(X_train_full))
print("Testing portion :", len(X_test))


Training portion: 2922
Testing portion : 731


In [ ]:
# CELL 11 — Validation split from training portion
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.10,
    random_state=42,
    stratify=y_train_full
)

print("Final split")
print("Train      :", len(X_train))
print("Validation :", len(X_val))
print("Test       :", len(X_test))


Final split
Train      : 2629
Validation : 293
Test       : 731


In [ ]:
# CELL 12 — Verify class distribution
print("Train distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nValidation distribution:")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTest distribution:")
print(pd.Series(y_test).value_counts().sort_index())


Train distribution:
0    687
1    705
2    690
3    547
Name: count, dtype: int64

Validation distribution:
0    77
1    78
2    77
3    61
Name: count, dtype: int64

Test distribution:
0    191
1    196
2    192
3    152
Name: count, dtype: int64


In [ ]:
# CELL 13 — Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(
    pd.DataFrame({"text": X_train.values, "label": y_train}),
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    pd.DataFrame({"text": X_val.values, "label": y_val}),
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    pd.DataFrame({"text": X_test.values, "label": y_test}),
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)


Dataset({
    features: ['text', 'label'],
    num_rows: 2629
})
Dataset({
    features: ['text', 'label'],
    num_rows: 293
})
Dataset({
    features: ['text', 'label'],
    num_rows: 731
})


In [ ]:
# CELL 14 — DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
DROPOUT = 0.1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_LENGTH)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Model: distilbert-base-uncased
Max sequence length: 128


In [ ]:
# CELL 15 — Tokenization
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenization completed.")
print(tokenized_train)


Map:   0%|          | 0/2629 [00:00<?, ? examples/s]

Map:   0%|          | 0/293 [00:00<?, ? examples/s]

Map:   0%|          | 0/731 [00:00<?, ? examples/s]

Tokenization completed.
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2629
})


In [ ]:
# CELL 16 — Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Data collator ready.")


Data collator ready.


In [ ]:
# CELL 17 — Evaluation metrics
# Accuracy + Macro Precision + Macro Recall + Macro F1

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:
# CELL 18 — Instructor's fixed settings
NUM_EPOCHS = 5
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 2

print("Epochs:", NUM_EPOCHS)
print("Warmup Ratio:", WARMUP_RATIO)
print("Max Sequence Length:", MAX_LENGTH)
print("Dropout:", DROPOUT)
print("Optimizer: AdamW")
print("Early stopping: enabled")
print("Patience:", EARLY_STOPPING_PATIENCE)


Epochs: 5
Warmup Ratio: 0.1
Max Sequence Length: 128
Dropout: 0.1
Optimizer: AdamW
Early stopping: enabled
Patience: 2


In [ ]:
# CELL 19 — EXACT 8 combinations from the instructor's table
COMBINATIONS = [
    {"combination": 1, "learning_rate": 0.00002, "batch_size": 16, "weight_decay": 0.01},
    {"combination": 2, "learning_rate": 0.00003, "batch_size": 16, "weight_decay": 0.01},
    {"combination": 3, "learning_rate": 0.00002, "batch_size": 32, "weight_decay": 0.01},
    {"combination": 4, "learning_rate": 0.00003, "batch_size": 32, "weight_decay": 0.01},
    {"combination": 5, "learning_rate": 0.00002, "batch_size": 16, "weight_decay": 0.1},
    {"combination": 6, "learning_rate": 0.00003, "batch_size": 16, "weight_decay": 0.1},
    {"combination": 7, "learning_rate": 0.00002, "batch_size": 32, "weight_decay": 0.1},
    {"combination": 8, "learning_rate": 0.00003, "batch_size": 32, "weight_decay": 0.1},
]

combination_df = pd.DataFrame(COMBINATIONS)
display(combination_df)


,combination,learning_rate,batch_size,weight_decay
0,1,0.00002,16,0.01
1,2,0.00003,16,0.01
2,3,0.00002,32,0.01
3,4,0.00003,32,0.01
4,5,0.00002,16,0.10
5,6,0.00003,16,0.10
6,7,0.00002,32,0.10
7,8,0.00003,32,0.10


In [ ]:
# CELL 20 — Output folder
BASE_OUTPUT_DIR = "/content/dataset2_distilbert_variation"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
print("Output directory:", BASE_OUTPUT_DIR)


Output directory: /content/dataset2_distilbert_variation


In [ ]:
# CELL 21 — Fresh DistilBERT model for every combination
def create_model():
    config = AutoConfig.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS
    )

    if hasattr(config, "seq_classif_dropout"):
        config.seq_classif_dropout = DROPOUT
    if hasattr(config, "dropout"):
        config.dropout = DROPOUT
    if hasattr(config, "attention_dropout"):
        config.attention_dropout = DROPOUT

    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config
    )


In [ ]:
# CELL 22 — Train one combination
def run_one_combination(combo):

    combo_id = combo["combination"]
    lr = combo["learning_rate"]
    batch_size = combo["batch_size"]
    weight_decay = combo["weight_decay"]

    print("\n" + "=" * 70)
    print(f"RUNNING COMBINATION {combo_id}/8")
    print(f"Learning Rate : {lr}")
    print(f"Batch Size    : {batch_size}")
    print(f"Weight Decay  : {weight_decay}")
    print("=" * 70)

    set_seed(SEED)
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = create_model()

    output_dir = os.path.join(
        BASE_OUTPUT_DIR,
        f"combination_{combo_id}"
    )

    # Exact equivalent of Warmup Ratio = 0.1
    steps_per_epoch = math.ceil(len(tokenized_train) / batch_size)
    total_steps = steps_per_epoch * NUM_EPOCHS
    warmup_steps = math.ceil(total_steps * WARMUP_RATIO)

    print("Steps per epoch:", steps_per_epoch)
    print("Total training steps:", total_steps)
    print("Warmup steps (10%):", warmup_steps)

    args_kwargs = dict(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        weight_decay=weight_decay,
        num_train_epochs=NUM_EPOCHS,
        warmup_steps=warmup_steps,
        optim="adamw_torch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_strategy="epoch",
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        remove_unused_columns=True
    )

    # Compatible with older/newer Transformers versions
    ta_params = inspect.signature(TrainingArguments.__init__).parameters

    if "eval_strategy" in ta_params:
        args_kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in ta_params:
        args_kwargs["evaluation_strategy"] = "epoch"
    else:
        raise RuntimeError(
            "This Transformers version has no evaluation strategy argument."
        )

    training_args = TrainingArguments(**args_kwargs)

    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE
            )
        ]
    )

    trainer_params = inspect.signature(Trainer.__init__).parameters

    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    train_result = trainer.train()

    validation_metrics = trainer.evaluate(
        eval_dataset=tokenized_val
    )

    result = {
        "combination": combo_id,
        "learning_rate": lr,
        "batch_size": batch_size,
        "weight_decay": weight_decay,
        "best_validation_accuracy": validation_metrics.get("eval_accuracy"),
        "best_validation_precision": validation_metrics.get("eval_precision"),
        "best_validation_recall": validation_metrics.get("eval_recall"),
        "best_validation_f1": validation_metrics.get("eval_f1"),
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "train_runtime_seconds": train_result.metrics.get("train_runtime")
    }

    os.makedirs(output_dir, exist_ok=True)

    with open(
        os.path.join(output_dir, "validation_result.json"),
        "w"
    ) as f:
        json.dump(result, f, indent=4)

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    print("\nCombination completed.")
    print("Validation Accuracy :", result["best_validation_accuracy"])
    print("Validation Precision:", result["best_validation_precision"])
    print("Validation Recall   :", result["best_validation_recall"])
    print("Validation F1       :", result["best_validation_f1"])

    del trainer
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


In [ ]:
# CELL 23 — Run all 8 combinations
# This cell performs the actual 8 training runs.

all_results = []

for combo in COMBINATIONS:

    result = run_one_combination(combo)
    all_results.append(result)

    current_results = pd.DataFrame(all_results)

    display(
        current_results[
            [
                "combination",
                "learning_rate",
                "batch_size",
                "weight_decay",
                "best_validation_accuracy",
                "best_validation_precision",
                "best_validation_recall",
                "best_validation_f1"
            ]
        ]
    )

print("\n" + "=" * 70)
print("ALL 8 COMBINATIONS COMPLETED.")
print("=" * 70)



RUNNING COMBINATION 1/8
Learning Rate : 2e-05
Batch Size    : 16
Weight Decay  : 0.01


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.032126,0.203694,5,0.938567,0.936557,0.938360,0.936825


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Combination completed.
Validation Accuracy : 0.9385665529010239
Validation Precision: 0.9365568641884432
Validation Recall   : 0.9383601371306289
Validation F1       : 0.9368249569291416


,combination,learning_rate,batch_size,weight_decay,best_validation_accuracy,best_validation_precision,best_validation_recall,best_validation_f1
0,1,0.00002,16,0.01,0.938567,0.937203,0.938360,0.937307
1,2,0.00003,16,0.01,0.938567,0.936763,0.939253,0.937238
2,3,0.00002,32,0.01,0.935154,0.932786,0.933369,0.932917
3,4,0.00003,32,0.01,0.938567,0.936557,0.938360,0.936825
4,5,0.00002,16,0.10,0.938567,0.937203,0.938360,0.937307
5,6,0.00003,16,0.10,0.938567,0.936763,0.939253,0.937238
6,7,0.00002,32,0.10,0.928328,0.925723,0.925255,0.924924
7,8,0.00003,32,0.10,0.938567,0.936557,0.938360,0.936825



ALL 8 COMBINATIONS COMPLETED.


In [ ]:
# CELL 24 — Final 8-combination validation table
results_df = (
    pd.DataFrame(all_results)
    .sort_values("combination")
    .reset_index(drop=True)
)

display(results_df)

RESULTS_CSV = os.path.join(
    BASE_OUTPUT_DIR,
    "dataset2_distilbert_8_combinations.csv"
)

results_df.to_csv(RESULTS_CSV, index=False)

print("Saved:", RESULTS_CSV)


,combination,learning_rate,batch_size,weight_decay,best_validation_accuracy,best_validation_precision,best_validation_recall,best_validation_f1,best_checkpoint,train_runtime_seconds
0,1,0.00002,16,0.01,0.938567,0.937203,0.938360,0.937307,/content/dataset2_distilbert_variation/combina...,99.7183
1,2,0.00003,16,0.01,0.938567,0.936763,0.939253,0.937238,/content/dataset2_distilbert_variation/combina...,120.4973
2,3,0.00002,32,0.01,0.935154,0.932786,0.933369,0.932917,/content/dataset2_distilbert_variation/combina...,91.2682
3,4,0.00003,32,0.01,0.938567,0.936557,0.938360,0.936825,/content/dataset2_distilbert_variation/combina...,74.8992
4,5,0.00002,16,0.10,0.938567,0.937203,0.938360,0.937307,/content/dataset2_distilbert_variation/combina...,101.3338
5,6,0.00003,16,0.10,0.938567,0.936763,0.939253,0.937238,/content/dataset2_distilbert_variation/combina...,173.4017
6,7,0.00002,32,0.10,0.928328,0.925723,0.925255,0.924924,/content/dataset2_distilbert_variation/combina...,87.5979
7,8,0.00003,32,0.10,0.938567,0.936557,0.938360,0.936825,/content/dataset2_distilbert_variation/combina...,149.6903


Saved: /content/dataset2_distilbert_variation/dataset2_distilbert_8_combinations.csv


In [ ]:
# CELL 25 — Select best combination using validation F1
best_row = results_df.loc[
    results_df["best_validation_f1"].idxmax()
]

print("BEST DISTILBERT COMBINATION")
print("=" * 50)
print("Combination :", int(best_row["combination"]))
print("Learning Rate:", best_row["learning_rate"])
print("Batch Size   :", int(best_row["batch_size"]))
print("Weight Decay :", best_row["weight_decay"])

print("\nValidation Accuracy :",
      best_row["best_validation_accuracy"])
print("Validation Precision:",
      best_row["best_validation_precision"])
print("Validation Recall   :",
      best_row["best_validation_recall"])
print("Validation F1       :",
      best_row["best_validation_f1"])

print("\nBest checkpoint:",
      best_row["best_checkpoint"])


BEST DISTILBERT COMBINATION
Combination : 1
Learning Rate: 2e-05
Batch Size   : 16
Weight Decay : 0.01

Validation Accuracy : 0.9385665529010239
Validation Precision: 0.937202601822855
Validation Recall   : 0.9383601371306289
Validation F1       : 0.937307210333526

Best checkpoint: /content/dataset2_distilbert_variation/combination_1/checkpoint-825


In [ ]:
# CELL 26 — Load the selected best model
best_checkpoint = best_row["best_checkpoint"]

best_tokenizer = AutoTokenizer.from_pretrained(
    best_checkpoint
)

best_model = AutoModelForSequenceClassification.from_pretrained(
    best_checkpoint
)

print("Best model loaded:", best_checkpoint)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Best model loaded: /content/dataset2_distilbert_variation/combination_1/checkpoint-825


In [ ]:
# CELL 27 — Tokenize test set using selected tokenizer
best_test_dataset = test_dataset.map(
    lambda examples: best_tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    ),
    batched=True,
    remove_columns=["text"]
)

best_collator = DataCollatorWithPadding(
    tokenizer=best_tokenizer
)

print(best_test_dataset)


Map:   0%|          | 0/731 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 731
})


In [ ]:
# CELL 28 — Final untouched test evaluation
test_args = TrainingArguments(
    output_dir="/content/dataset2_distilbert_test_eval",
    per_device_eval_batch_size=int(best_row["batch_size"]),
    report_to="none",
    fp16=torch.cuda.is_available()
)

test_trainer_kwargs = dict(
    model=best_model,
    args=test_args,
    data_collator=best_collator,
    compute_metrics=compute_metrics
)

test_trainer_params = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in test_trainer_params:
    test_trainer_kwargs["processing_class"] = best_tokenizer
elif "tokenizer" in test_trainer_params:
    test_trainer_kwargs["tokenizer"] = best_tokenizer

test_trainer = Trainer(**test_trainer_kwargs)

test_metrics = test_trainer.evaluate(
    eval_dataset=best_test_dataset
)

print("FINAL TEST RESULTS")
print("=" * 50)
print("Accuracy :", test_metrics["eval_accuracy"])
print("Precision:", test_metrics["eval_precision"])
print("Recall   :", test_metrics["eval_recall"])
print("F1       :", test_metrics["eval_f1"])


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
No log,0.309402,0,0.922025,0.921596,0.921503,0.921361


FINAL TEST RESULTS
Accuracy : 0.9220246238030095
Precision: 0.9215964520533049
Recall   : 0.9215031665738767
F1       : 0.9213608041305318


In [ ]:
# CELL 29 — Final Dataset-2 table row
final_results = pd.DataFrame([{
    "Model": "DistilBERT (BERT Variation)",
    "Dataset": "Dataset-2",
    "Learning Rate": best_row["learning_rate"],
    "Batch Size": int(best_row["batch_size"]),
    "Weight Decay": best_row["weight_decay"],
    "Acc": test_metrics["eval_accuracy"] * 100,
    "Prec": test_metrics["eval_precision"] * 100,
    "Rec": test_metrics["eval_recall"] * 100,
    "F1": test_metrics["eval_f1"] * 100
}])

display(final_results)


,Model,Dataset,Learning Rate,Batch Size,Weight Decay,Acc,Prec,Rec,F1
0,DistilBERT (BERT Variation),Dataset-2,0.00002,16,0.01,92.202462,92.159645,92.150317,92.13608


In [ ]:
# CELL 30 — Classification report and confusion matrix
pred_output = test_trainer.predict(best_test_dataset)

test_predictions = np.argmax(
    pred_output.predictions,
    axis=-1
)

test_true_labels = pred_output.label_ids

print(
    classification_report(
        test_true_labels,
        test_predictions,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(
    test_true_labels,
    test_predictions
)

cm_df = pd.DataFrame(
    cm,
    index=label_encoder.classes_,
    columns=label_encoder.classes_
)

print("Confusion Matrix:")
display(cm_df)


                    precision    recall  f1-score   support

     comp.graphics     0.9560    0.9110    0.9330       191
  rec.sport.hockey     0.9343    0.9439    0.9391       196
         sci.space     0.8934    0.9167    0.9049       192
talk.politics.misc     0.9026    0.9145    0.9085       152

          accuracy                         0.9220       731
         macro avg     0.9216    0.9215    0.9214       731
      weighted avg     0.9227    0.9220    0.9221       731

Confusion Matrix:


,comp.graphics,rec.sport.hockey,sci.space,talk.politics.misc
comp.graphics,174,7,7,3
rec.sport.hockey,0,185,5,6
sci.space,7,3,176,6
talk.politics.misc,1,3,9,139


In [ ]:
# CELL 31 — Save test probabilities for later ensemble
test_probabilities = torch.softmax(
    torch.tensor(pred_output.predictions),
    dim=1
).numpy()

prediction_output = pd.DataFrame(
    test_probabilities,
    columns=[
        f"prob_{label}"
        for label in label_encoder.classes_
    ]
)

prediction_output["true_label"] = test_true_labels
prediction_output["predicted_label"] = test_predictions

PREDICTION_PATH = os.path.join(
    BASE_OUTPUT_DIR,
    "best_distilbert_test_predictions.csv"
)

prediction_output.to_csv(
    PREDICTION_PATH,
    index=False
)

FINAL_RESULTS_PATH = os.path.join(
    BASE_OUTPUT_DIR,
    "best_distilbert_final_results.csv"
)

final_results.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)

print("Test probabilities:", PREDICTION_PATH)
print("Final results:", FINAL_RESULTS_PATH)


Test probabilities: /content/dataset2_distilbert_variation/best_distilbert_test_predictions.csv
Final results: /content/dataset2_distilbert_variation/best_distilbert_final_results.csv
